# ProstT5 Speculative Decoding with CNN Drafter (3Di → AA)

This notebook benchmarks **speculative decoding** for ProstT5 inverse folding using a CNN drafter.

**Structure:**
- **Setup** — imports, model loading, dataset, hyperparameters
- **Part 1** — Greedy Speculative Decoding: custom implementation, sweep over K values
- **Part 2** — HuggingFace `assistant_model` integration: verify outputs match custom implementation

In [ ]:
%pip install -q tiktoken sentencepiece torch
%pip install -q 'accelerate>=0.26.0'
%pip install -q "transformers==4.46.3" "protobuf>=3.20,<5" sentencepiece
%pip install -q datasets pandas matplotlib

!pip install --upgrade pip
!pip install --upgrade "protobuf<=3.20.3" transformers sentencepiece

## 1. Setup

In [2]:
# @title 1. Setup, Imports, and Hardware Check
import os, time, statistics, gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, GenerationConfig

try:
    from google.colab import drive
    drive.mount('/content/drive')
    USE_DRIVE = True
    DRIVE_ROOT = '/content/drive/MyDrive'
except Exception:
    USE_DRIVE = False
    DRIVE_ROOT = None

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}  |  Device: {device}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.10.0+cpu  |  Device: cpu


In [ ]:
# @title [Optional] GPU Memory Cleanup — run this before re-running the notebook
# If you see CUDA OOM errors from a previous run in the same Colab session,
# run this cell first to free GPU memory, then re-run from cell 2 onward.
import gc, torch

_to_clear = ['model', 'encoder', 'cnn', 'cnn_assistant',
             'part1', 'spec_results', 'hf_results', 'df1', 'df2', 'df3']
for _v in _to_clear:
    if _v in globals():
        del globals()[_v]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory cleared.  Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB total")
else:
    print("No CUDA GPU — nothing to clear.")
print("Re-run cells from '2. Configuration & Model Loading' onward.")

No CUDA GPU — nothing to clear.
Re-run cells from '2. Configuration & Model Loading' onward.


: 

## 2. Configuration & Model Loading

Upload the CNN checkpoint (`model.pt`) to Google Drive or place it next to this notebook.

In [4]:
# @title 2. Configuration & Model Loading
PROSTT5_NAME  = "Rostlab/ProstT5_fp16"
NOTEBOOK_DIR  = Path(".").resolve()
AA_LETTERS    = "ACDEFGHIKLMNPQRSTVWY"  # 20 standard AAs, alphabetical (matches CNN output)
NUM_WARMUP    = 1   # warmup passes before timing
NUM_REPEATS   = 3   # timed repeats; report median

# Plot colors (defined here so all cells work without re-running setup)
BLUE   = "#2e86de"
ORANGE = "#f39c12"
GRAY   = "#7f8c8d"

# Locate CNN checkpoint
CNN_CKPT = None
for candidate in [
    NOTEBOOK_DIR / "model.pt",
    NOTEBOOK_DIR / "cnn_chkpnt_AA_CNN" / "model.pt",
    *([] if not USE_DRIVE else [
        Path(DRIVE_ROOT) / "models" / "CNN_from3di_toAA.pt",
        Path(DRIVE_ROOT) / "model.pt",
    ]),
]:
    if candidate.exists():
        CNN_CKPT = candidate
        break

if CNN_CKPT is None:
    raise FileNotFoundError("CNN checkpoint not found. Upload model.pt to the notebook directory.")
print(f"CNN checkpoint: {CNN_CKPT}")

# ── Tokenizer (skip if already in memory) ──────────────────────────────────
if 'tokenizer' not in globals():
    print("Loading tokenizer...")
    tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)
else:
    print("Tokenizer already in memory, skipping load.")

# ── ProstT5 model (5.6 GB — skip if already in memory to avoid OOM) ────────
if 'model' not in globals():
    print("Loading ProstT5 model (~5.6 GB)...")
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(
        PROSTT5_NAME, low_cpu_mem_usage=True, torch_dtype=dtype,
    ).to(device).eval()
    print(f"ProstT5: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
else:
    print("ProstT5 already in memory, skipping load.")
encoder = model.get_encoder()

# ── CNN architecture (class definition is cheap; always safe to redefine) ───
class AACNN(nn.Module):
    def __init__(self, num_tokens=20, hidden=32, in_dim=1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )
    def forward(self, x):
        x = x.permute(0, 2, 1).unsqueeze(-1)
        x = self.classifier(x)
        return x.squeeze(-1).permute(0, 2, 1)

if 'cnn' not in globals():
    cnn = AACNN().to(device).eval()
    ckpt = torch.load(CNN_CKPT, map_location=device, weights_only=False)
    cnn.load_state_dict(ckpt.get("state_dict", ckpt), strict=True)
    print(f"CNN loaded: {sum(p.numel() for p in cnn.parameters()):,} params")
else:
    print("CNN already in memory, skipping load.")

# ── Token ID mappings (fast to recompute, always run) ──────────────────────
# ProstT5 generates space-separated AAs; tokens are ▁A, ▁C, ... so encode(" A")
AA_TOKEN_IDS           = [tokenizer.encode(f" {aa}", add_special_tokens=False)[0] for aa in AA_LETTERS]
CNN_IDX_TO_TOKEN_ID    = {i: tid for i, tid in enumerate(AA_TOKEN_IDS)}
AA_TOKEN_ID_TO_CNN_IDX = {tid: i for i, tid in enumerate(AA_TOKEN_IDS)}
DECODER_START_TOKEN_ID = model.config.decoder_start_token_id

# Sanity check
for aa, tid in zip(AA_LETTERS[:3], AA_TOKEN_IDS[:3]):
    decoded = tokenizer.decode([tid]).strip()
    assert decoded == aa, f"Token mismatch: {aa} -> {tid} -> '{decoded}'"

print(f"Decoder start token ID: {DECODER_START_TOKEN_ID}")
print(f"Example AA token IDs: {list(zip(AA_LETTERS[:5], AA_TOKEN_IDS[:5]))}")
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {used:.1f} GB used / {total:.1f} GB total")

CNN checkpoint: /content/drive/MyDrive/models/CNN_from3di_toAA.pt
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading ProstT5 model (~5.6 GB)...


model.safetensors:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

: 

: 

## 3. Dataset — 100-Protein Test Set

In [ ]:
# @title 3. Load 100-Protein FASTA Test Set
def parse_fasta(path):
    seqs = {}
    cur  = None
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            seqs[cur] = ""
        elif cur:
            seqs[cur] += line.strip()
    return seqs

_drive_candidates = [] if not USE_DRIVE else [
    Path(DRIVE_ROOT) / "models",                                          # same folder as CNN checkpoint
    Path(DRIVE_ROOT) / "benchmark_data",
    Path(DRIVE_ROOT) / "prostT5" / "benchmark_data",
    Path(DRIVE_ROOT) / "Speculative-Decoding-ProstT5" / "prostT5" / "benchmark_data",
]
DATA_DIR = None
for candidate in [
    NOTEBOOK_DIR / "benchmark_data",
    Path("/content/benchmark_data"),
    Path("/content/prostT5/benchmark_data"),
    *_drive_candidates,
]:
    if (candidate / "test_set_AA.fasta").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "test_set_AA.fasta not found. Upload both FASTA files to MyDrive/models/ "
        "(same folder as CNN_from3di_toAA.pt)."
    )

aa_seqs = parse_fasta(DATA_DIR / "test_set_AA.fasta")
di_seqs = parse_fasta(DATA_DIR / "test_set_3Di.fasta")

benchmark_data = [
    {"id": uid, "AA": aa_seqs[uid], "3Di": di_seqs[uid].lower()}
    for uid in aa_seqs if uid in di_seqs
]
lengths = [len(r["AA"]) for r in benchmark_data]

print(f"Loaded {len(benchmark_data)} proteins from {DATA_DIR}")
print(f"Length range: {min(lengths)}–{max(lengths)} AA  |  median: {sorted(lengths)[len(lengths)//2]}")
print(f"First 3: {[(r['id'], len(r['AA'])) for r in benchmark_data[:3]]}")

Loaded 97 proteins from /content/drive/MyDrive/models
Length range: 46–2554 AA  |  median: 427
First 3: [('P01308', 110), ('P02798', 61), ('P62944', 937)]


In [ ]:
# @title Hyperparameters — edit these before running
# ── Speculative decoding sweep ──────────────────────────────────────────────
K_VALUES     = [4]   # draft lengths to sweep in Part 1

# ── Dataset ─────────────────────────────────────────────────────────────────
NUM_PROTEINS = 10         # None = all proteins; set to an int to limit (e.g. 10 for a quick test)

# ── Timing protocol ─────────────────────────────────────────────────────────
SPEC_WARMUP  = 1            # warmup passes before timing (discarded)
SPEC_REPEATS = 3            # timed repeats; median is reported

# ── HuggingFace assisted generation (Part 2) ────────────────────────────────
HF_K         = 8            # num_assistant_tokens for HF assisted generation

# Apply dataset limit
if NUM_PROTEINS is not None:
    benchmark_data = benchmark_data[:NUM_PROTEINS]
    print(f"Dataset limited to {NUM_PROTEINS} proteins.")
else:
    print(f"Using all {len(benchmark_data)} proteins.")
print(f"K_VALUES={K_VALUES}  |  HF_K={HF_K}  |  SPEC_WARMUP={SPEC_WARMUP}  |  SPEC_REPEATS={SPEC_REPEATS}")

Using all 97 proteins.
K_VALUES=[8]  |  HF_K=8  |  SPEC_WARMUP=1  |  SPEC_REPEATS=3


## Part 1 — Greedy Speculative Decoding (Custom Implementation)

Implements Leviathan et al.'s speculative decoding:
- The **CNN drafter** proposes *K* tokens in one shot (prefix-independent).
- The **enc-dec verifier** checks all *K* tokens in a single forward pass.
- Accept/reject greedily — a **bonus token** is extracted when all *K* drafts are accepted.
- Sweep over `K_VALUES` to measure acceptance rate and tokens per step.

In [ ]:
# @title Part 1A — Helper functions & speculative_decode_greedy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

def _sync():
    if device.type == "cuda":   torch.cuda.synchronize()
    elif device.type == "mps":  torch.mps.synchronize()

def _format_3di(seq):
    return "<fold2AA> " + " ".join(list(seq.lower()))

def _decode_aa(token_ids):
    s = tokenizer.decode(token_ids if isinstance(token_ids, list) else token_ids.tolist(),
                         skip_special_tokens=True)
    return "".join(s.split())


@torch.inference_mode()
def speculative_decode_greedy(three_di: str, K: int = 5) -> dict:
    '''
    Greedy speculative decoding with CNN drafter + enc-dec verifier.
    Returns a dict with sequence, timing, and acceptance statistics.
    '''
    L = len(three_di)

    # 1. Encode once (shared between CNN and decoder)
    enc_input    = tokenizer([_format_3di(three_di)], add_special_tokens=True,
                             return_tensors="pt").to(device)
    encoder_out  = encoder(input_ids=enc_input.input_ids,
                           attention_mask=enc_input.attention_mask)
    enc_hidden   = encoder_out.last_hidden_state          # (1, seq_len, 1024)

    # 2. CNN draft logits — all positions at once (prefix-independent)
    h_cnn        = enc_hidden[:, 1:-1, :]                 # trim prefix + EOS tokens
    cnn_logits   = cnn(h_cnn.float())[0]                 # (L, 20)

    # Speculative decoding loop
    generated    = [DECODER_START_TOKEN_ID]               # starts with decoder_start
    accepted_counts = []
    k_history    = []
    n_steps      = 0

    _sync()
    t_start = time.perf_counter()

    while len(generated) - 1 < L:
        pos = len(generated) - 1
        k   = min(K, L - pos)
        if k <= 0:
            break
        k_history.append(k)

        # Draft: CNN argmax at positions [pos, pos+k)
        draft_cnn_idx  = cnn_logits[pos:pos + k].argmax(dim=-1)      # (k,)
        draft_tok_ids  = [CNN_IDX_TO_TOKEN_ID[int(i)] for i in draft_cnn_idx]

        # Verify: single decoder forward pass over the drafted sequence
        verify_input = torch.tensor([generated + draft_tok_ids],
                                    device=device, dtype=torch.long)
        dec_out      = model(encoder_outputs=(enc_hidden,),
                             decoder_input_ids=verify_input)
        ver_logits   = dec_out.logits[0]                 # (len(generated)+k, vocab)

        # Accept / reject greedily
        n_accepted = 0
        for i in range(k):
            ver_tok = int(ver_logits[pos + i].argmax())
            if draft_tok_ids[i] == ver_tok:
                n_accepted += 1
            else:
                # Reject: take tokens up to mismatch + verifier's correction
                generated.extend(draft_tok_ids[:n_accepted])
                generated.append(ver_tok)
                break
        else:
            # All K accepted → also take bonus token
            generated.extend(draft_tok_ids)
            if len(generated) - 1 < L:
                bonus_pos = pos + k
                if bonus_pos < ver_logits.shape[0]:
                    generated.append(int(ver_logits[bonus_pos].argmax()))

        accepted_counts.append(n_accepted)
        n_steps += 1

    _sync()
    wall_time = time.perf_counter() - t_start

    output_ids = generated[1:]
    output_aa  = _decode_aa(output_ids)[:L]
    toks_per_step = [a + 1 for a in accepted_counts]

    return dict(
        sequence          = output_aa,
        wall_time         = wall_time,
        n_steps           = n_steps,
        K                 = K,
        accepted_counts   = accepted_counts,
        k_history         = k_history,
        mean_accepted     = float(np.mean(accepted_counts)) if accepted_counts else 0.0,
        tokens_per_step   = toks_per_step,
        mean_tps          = float(np.mean(toks_per_step)) if toks_per_step else 0.0,
        acceptance_rate   = (float(np.mean(accepted_counts)) / np.mean(k_history)
                             if k_history else 0.0),
    )

print("speculative_decode_greedy defined.")

speculative_decode_greedy defined.


In [ ]:
# @title Part 1B — Benchmark spec decode across all proteins (K sweep)
@torch.inference_mode()
def time_spec_decode(three_di, K):
    '''Time speculative decoding: warmup runs then median over SPEC_REPEATS.'''
    for _ in range(SPEC_WARMUP):
        speculative_decode_greedy(three_di, K=K)
    _sync()
    times = []; last = None
    for _ in range(SPEC_REPEATS):
        res = speculative_decode_greedy(three_di, K=K)
        times.append(res["wall_time"]); last = res
    return statistics.median(times), last

print(f"Sweeping K = {K_VALUES} across {len(benchmark_data)} proteins...")
print(f"Protocol: {SPEC_WARMUP} warmup + {SPEC_REPEATS} repeats, median\n")

spec_results = []  # one row per (protein, K)

for item in tqdm(benchmark_data, desc="Proteins"):
    uid, three_di = item["id"], item["3Di"]
    L = len(three_di)

    for K in K_VALUES:
        t_spec, last_res = time_spec_decode(three_di, K)
        spec_results.append(dict(
            id              = uid,
            length          = L,
            K               = K,
            t_spec          = t_spec,
            acceptance_rate = last_res["acceptance_rate"],
            mean_tps        = last_res["mean_tps"],
            n_steps         = last_res["n_steps"],
        ))

import pandas as pd
df2 = pd.DataFrame(spec_results)
print("\nSpec decode benchmark complete.")
print("\nMean stats by K:")
print(df2.groupby("K")[["acceptance_rate", "mean_tps", "n_steps"]].mean().round(3).to_string())

Sweeping K = [8] across 97 proteins...
Protocol: 1 warmup + 3 repeats, median



Proteins:   0%|          | 0/97 [00:00<?, ?it/s]

In [ ]:
# @title Part 1C — Spec Decode Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Part 1 — Greedy Speculative Decoding (CNN Drafter)", fontsize=14, y=1.01)
axes = axes.flatten()
PALETTE = {1: "#3498db", 4: "#2ecc71", 8: "#e74c3c"}

# ── 1. Wall-time distribution by K ──────────────────────────────────────────
ax = axes[0]
data_wt = [df2[df2["K"] == K]["t_spec"].values for K in K_VALUES]
bp = ax.boxplot(data_wt, patch_artist=True, widths=0.5,
                medianprops={"color": "black", "lw": 2})
for patch, K in zip(bp["boxes"], K_VALUES):
    patch.set_facecolor(PALETTE[K]); patch.set_alpha(0.7)
ax.set_xticklabels([f"K={K}" for K in K_VALUES])
ax.set_ylabel("Wall time (s)"); ax.set_title("Spec Decode Wall Time vs K")

# ── 2. Acceptance rate vs K ──────────────────────────────────────────────────
ax = axes[1]
data_ar = [df2[df2["K"] == K]["acceptance_rate"].dropna().values for K in K_VALUES]
bp2 = ax.boxplot(data_ar, patch_artist=True, widths=0.5,
                 medianprops={"color": "black", "lw": 2})
for patch, K in zip(bp2["boxes"], K_VALUES):
    patch.set_facecolor(PALETTE[K]); patch.set_alpha(0.7)
ax.set_xticklabels([f"K={K}" for K in K_VALUES])
ax.set_ylabel("Acceptance rate (α)"); ax.set_title("Token Acceptance Rate vs K")
ax.set_ylim(0, 1.05)

# ── 3. Measured vs predicted tokens per step ────────────────────────────────
ax = axes[2]
mean_tps   = [df2[df2["K"] == K]["mean_tps"].mean() for K in K_VALUES]
mean_alpha = [df2[df2["K"] == K]["acceptance_rate"].mean() for K in K_VALUES]
pred_tps   = [(1 - a**(K+1)) / (1 - a) if a < 0.9999 else K + 1
              for a, K in zip(mean_alpha, K_VALUES)]
x = np.arange(len(K_VALUES))
w = 0.35
ax.bar(x - w/2, mean_tps,  w, color=BLUE,   alpha=0.8, label="Measured")
ax.bar(x + w/2, pred_tps,  w, color=ORANGE, alpha=0.8, label="Predicted (Thm 3.8)")
ax.set_xticks(x); ax.set_xticklabels([f"K={K}" for K in K_VALUES])
ax.set_ylabel("Mean tokens per step"); ax.set_title("Measured vs Predicted Tokens/Step")
ax.legend(fontsize=9)

# ── 4. Acceptance rate vs protein length ────────────────────────────────────
ax = axes[3]
best_K = K_VALUES[len(K_VALUES) // 2]
dKbest = df2[df2["K"] == best_K]
sc = ax.scatter(dKbest["length"], dKbest["acceptance_rate"], c=dKbest["t_spec"],
                cmap="YlOrRd", s=40, alpha=0.8, edgecolors="none")
ax.set_xlabel("Protein length (AA)"); ax.set_ylabel("Acceptance rate (α)")
ax.set_title(f"Acceptance Rate vs Length (K={best_K}, colored by wall time)")
ax.set_ylim(0, 1.05)
fig.colorbar(sc, ax=ax, label="Wall time (s)")

plt.tight_layout()
plt.savefig("part1_spec_decode.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: part1_spec_decode.png")

## Part 2 — HuggingFace `assistant_model` Integration

Wraps the CNN as a `PreTrainedModel` so we can use `model.generate(..., assistant_model=...)`.
This is HuggingFace's built-in speculative decoding — we verify its output matches the custom implementation above.

In [ ]:
# @title Part 3A — CNNAssistantModel wrapper
from transformers import PreTrainedModel
from transformers.modeling_outputs import Seq2SeqLMOutput

class CNNAssistantModel(PreTrainedModel):
    '''Wraps enc-CNN as a HF-compatible assistant model for model.generate().
    The CNN is prefix-independent: it always produces the same logits for a given
    encoder input. We pre-compute all logits once and serve slices on demand.
    '''
    config_class = type(model.config)

    def __init__(self, config, prostt5_encoder, cnn_head, aa_token_ids):
        super().__init__(config)
        self._encoder      = prostt5_encoder
        self._cnn          = cnn_head
        self._aa_token_ids = aa_token_ids
        self.config.is_encoder_decoder         = True
        self.config.decoder_start_token_id     = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
        )
        self._cached_logits     = None
        self._cached_hidden_id  = None

    def _full_vocab_logits(self, encoder_outputs):
        hidden = encoder_outputs[0]          # (1, seq_len, 1024)
        h      = hidden[:, 1:-1, :]          # trim prefix + EOS
        logits20 = self._cnn(h.float())[0]  # (L, 20)
        vocab_size = self.config.vocab_size
        full = torch.full((logits20.shape[0], vocab_size), -1e4,
                          device=logits20.device, dtype=logits20.dtype)
        for cnn_idx, tok_id in enumerate(self._aa_token_ids):
            full[:, tok_id] = logits20[:, cnn_idx]
        return full                          # (L, vocab)

    def get_encoder(self):
        return self._encoder

    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kw):
        return {"decoder_input_ids": decoder_input_ids, "encoder_outputs": encoder_outputs}

    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kw):
        if encoder_outputs is not None:
            h_id = id(encoder_outputs[0])
            if self._cached_hidden_id != h_id:
                self._cached_logits    = self._full_vocab_logits(encoder_outputs)
                self._cached_hidden_id = h_id
        n = decoder_input_ids.shape[1]
        if self._cached_logits is not None:
            L_cached = self._cached_logits.shape[0]
            n_ret    = min(n, L_cached)
            logits   = self._cached_logits[:n_ret].unsqueeze(0)
            if n_ret < n:
                pad    = torch.full((1, n - n_ret, logits.shape[-1]), -1e4,
                                    device=logits.device, dtype=logits.dtype)
                logits = torch.cat([logits, pad], dim=1)
        else:
            logits = torch.zeros(1, n, self.config.vocab_size, device=device)
        return Seq2SeqLMOutput(logits=logits)

# Instantiate
cnn_assistant = CNNAssistantModel(
    config         = model.config,
    prostt5_encoder= encoder,
    cnn_head       = cnn,
    aa_token_ids   = AA_TOKEN_IDS,
).to(device).eval()

print("CNNAssistantModel created.")
print(f"  is_encoder_decoder    = {cnn_assistant.config.is_encoder_decoder}")
print(f"  num_assistant_tokens  = {cnn_assistant.generation_config.num_assistant_tokens}")

In [ ]:
# @title Part 2B — Run HF assisted generation on all proteins
@torch.inference_mode()
def run_hf_assisted(three_di, num_assistant_tokens=5):
    '''Run model.generate with CNNAssistantModel; return (predicted_AA, median_wall_s).'''
    L   = len(three_di)
    inp = tokenizer([_format_3di(three_di)], add_special_tokens=True,
                    return_tensors="pt").to(device)
    cnn_assistant.generation_config.num_assistant_tokens = num_assistant_tokens
    kw  = dict(input_ids=inp.input_ids, attention_mask=inp.attention_mask,
               max_length=L + 2, do_sample=False, num_beams=1,
               assistant_model=cnn_assistant)
    for _ in range(NUM_WARMUP):
        model.generate(**kw)
    _sync()
    times = []
    for _ in range(NUM_REPEATS):
        _sync(); t0 = time.perf_counter()
        out = model.generate(**kw)
        _sync(); times.append(time.perf_counter() - t0)
    pred = _decode_aa(out[0])[:L]
    return pred, statistics.median(times)

print(f"Running HF assisted generation (num_assistant_tokens={HF_K}) on all proteins...")

hf_results = []
for item in tqdm(benchmark_data, desc="HF assisted"):
    uid, three_di = item["id"], item["3Di"]
    L = len(three_di)
    try:
        pred_hf, t_hf = run_hf_assisted(three_di, num_assistant_tokens=HF_K)
        hf_ok = True
    except Exception as exc:
        print(f"  {uid}: HF failed — {exc}")
        pred_hf, t_hf, hf_ok = "", float("nan"), False

    res_custom  = speculative_decode_greedy(three_di, K=HF_K)
    pred_custom = res_custom["sequence"]

    t_spec_k = next((r["t_spec"] for r in spec_results
                     if r["id"] == uid and r["K"] == HF_K), float("nan"))

    hf_results.append(dict(
        id                = uid,
        length            = L,
        hf_ok             = hf_ok,
        t_hf              = t_hf,
        t_custom_spec     = t_spec_k,
        acceptance_rate   = res_custom["acceptance_rate"],
        mean_tps          = res_custom["mean_tps"],
        n_steps           = res_custom["n_steps"],
        hf_matches_custom = (pred_hf == pred_custom) if hf_ok else False,
    ))

import pandas as pd
df3 = pd.DataFrame(hf_results)
n_ok    = int(df3["hf_ok"].sum())
n_hf_cu = int(df3["hf_matches_custom"].sum())

print(f"\nHF generation succeeded:       {n_ok}/{len(df3)} proteins")
print(f"HF matches custom spec-decode: {n_hf_cu}/{n_ok} of successful")
print(f"\nMean stats (K={HF_K}):")
print(df3[["acceptance_rate", "mean_tps", "n_steps"]].mean().round(3).to_string())

In [ ]:
# @title Part 2C — Comparison plots + final summary
_n_ok    = int(df3["hf_ok"].sum())
_n_hf_cu = int(df3["hf_matches_custom"].sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Part 2 — HF assistant_model vs Custom Spec Decode (K={HF_K})", fontsize=13)

# ── 1. Wall-time comparison (HF vs custom) ──────────────────────────────────
ax = axes[0]
t_hf_vals = df3["t_hf"].dropna().values
t_cu_vals = df3["t_custom_spec"].dropna().values
bp = ax.boxplot([t_cu_vals, t_hf_vals], patch_artist=True, widths=0.5,
                medianprops={"color": "black", "lw": 2})
for patch, c in zip(bp["boxes"], [BLUE, ORANGE]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_xticklabels([f"Custom spec-decode\n(K={HF_K})", f"HF assistant\n(K={HF_K})"], fontsize=9)
ax.set_ylabel("Wall time (s)")
ax.set_title("Wall Time: HF vs Custom Spec-Decode")

# ── 2. Output agreement ──────────────────────────────────────────────────────
ax = axes[1]
n_total = len(df3)
categories = [f"HF succeeded\n({n_total} total)", f"HF matches\ncustom\n(of {_n_ok} ok)"]
counts     = [_n_ok, _n_hf_cu]
totals     = [n_total, _n_ok]
pcts       = [c / t * 100 if t > 0 else 0 for c, t in zip(counts, totals)]
bars = ax.bar(categories, pcts, color=[BLUE, GRAY], alpha=0.8, edgecolor="white")
ax.set_ylim(0, 115)
ax.set_ylabel("Rate (%)")
ax.set_title("HF Generation Success & Output Agreement")
for bar, pct, cnt, tot in zip(bars, pcts, counts, totals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f"{pct:.1f}%\n({cnt}/{tot})", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("part2_hf_comparison.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: part2_hf_comparison.png")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL SUMMARY — Speculative Decoding Benchmark")
print("="*65)
print(f"{'Method':<35} {'Accept rate (α)':>16} {'Toks/step':>10} {'Steps (med)':>12}")
print("-"*65)
for K in K_VALUES:
    ar  = df2[df2["K"] == K]["acceptance_rate"].mean()
    tps = df2[df2["K"] == K]["mean_tps"].mean()
    ns  = df2[df2["K"] == K]["n_steps"].median()
    print(f"{'Spec decode  K='+str(K):<35} {ar:>16.3f} {tps:>10.2f} {ns:>12.0f}")
print("-"*65)
ar_hf  = df3["acceptance_rate"].mean()
tps_hf = df3["mean_tps"].mean()
ns_hf  = df3["n_steps"].median()
print(f"{'HF assistant  K='+str(HF_K):<35} {ar_hf:>16.3f} {tps_hf:>10.2f} {ns_hf:>12.0f}")
print("="*65)
match_str = (f"{_n_hf_cu}/{_n_ok} ({_n_hf_cu/_n_ok*100:.1f}%)" if _n_ok > 0 else "N/A")
print(f"\nHF output matches custom spec-decode: {match_str} of successful proteins")